# (노트) 텐서보드 사용방법1

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [데이터과학]

### imports

In [1]:
import tensorflow as tf

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

### data

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

In [5]:
X = x_train.reshape(-1,28,28,1)/255
y = tf.keras.utils.to_categorical(y_train)
XX = x_test.reshape(-1,28,28,1)/255
yy = tf.keras.utils.to_categorical(y_test)

### 텐서보드 기본시각화 

In [ ]:
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')

In [ ]:
#collapse
!rm -rf logs
cb1 = tf.keras.callbacks.TensorBoard()
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=cb1)

`-` 파이썬 3.10의 경우 아래의 수정이 필요

`?/python3.10/site-packages/tensorboard/_vendor/html5lib/_trie/_base.py` 을 열고
```python
from collections import Mapping ### 수정전
from collections.abc import Mapping ### 수정후 
```
와 같이 수정한다. 

- 에러나는이유? 파이썬 3.10부터는 `from collections.abc import Mapping`이 실행되고 `from collections import Mapping`은 실행안된다. 

`-` 텐서보드 여는 방법1

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs --host 0.0.0.0

- 노트북에서 바로보인다..

`-` 텐서보드 여는 방법2 

In [ ]:
!tensorboard --logdir logs --host 0.0.0.0

- 이제 `210.117.173.184:6007` 로 가면된다. 

### 텐서보드에서 그래프 보는 방법

`-` 그래프에서 tag를 keras로 변경하면 우리의 아키텐처가 간단하게 보인다. (지금은 이게 너무 간단하여 쓸모없어 보인다) 

### 조기종료 

`-` 텐서보드를 살펴보니 50에폭 이후로는 과적합이 진행되는 듯 하다. -> 50번까지만 학습한 모형이 있다면 그게 젤 좋은 것 같다. 

In [ ]:
cb2 = tf.keras.callbacks.EarlyStopping() # 조기종료 콜백생성 

`-` 이미학습한 네트워크를 초기화 

In [ ]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=cb2)

- 2번 돌고 멈추었다? 

In [ ]:
cb2 = tf.keras.callbacks.EarlyStopping(patience=1) # 조기종료 콜백생성 - val_loss가 증가하는 순간 멈춘다. 

In [ ]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=cb2)

In [ ]:
cb2 = tf.keras.callbacks.EarlyStopping(patience=2) # 조기종료 콜백생성 - val_loss가 증가하는 순간 멈추는데 1번은 봐줌

In [ ]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(1000,activation='relu')) 
net.add(tf.keras.layers.Dense(1000,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=60000,validation_split=0.2,callbacks=cb2,verbose=2)

`-` 조기종료콜백과 텐서보드콜백을 같이 쓰고싶다면? 

In [ ]:
tf.random.set_seed(43052)
net = tf.keras.Sequential()
net.add(tf.keras.layers.Flatten()) 
net.add(tf.keras.layers.Dense(50,activation='relu')) 
net.add(tf.keras.layers.Dense(10,activation='softmax')) 
net.compile(optimizer='adam',loss=tf.losses.categorical_crossentropy,metrics='accuracy')
net.fit(X,y,epochs=200,batch_size=200,validation_split=0.2,callbacks=[cb1,cb2],verbose=0)

In [ ]:
#%tensorboard --logdir logs --host 0.0.0.0

### 하이퍼파라메터 튜닝

`-` 참고문헌

- https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams

`-` 하이퍼파라메터 설정

In [6]:
from tensorboard.plugins.hparams import api as hp

In [8]:
!rm -rf logs
for u in [200,2000]:
    for d in [0.0,0.8]: 
        for o in ['adam','sgd']:
            #hparams = {HP_NUM_UNITS: num_units, HP_DROPOUT: dropout_rate, HP_OPTIMIZER: optimizer,}
            logdir = 'logs/hparam_tuning_{}_{}_{}'.format(u,d,o)
            with tf.summary.create_file_writer(logdir).as_default():
                #hp.hparams(hparams)          
                net = tf.keras.Sequential()
                net.add(tf.keras.layers.Flatten())
                net.add(tf.keras.layers.Dense(u, activation='relu'))
                net.add(tf.keras.layers.Dropout(d))
                net.add(tf.keras.layers.Dense(10, activation='softmax'))
                net.compile(optimizer=o,loss=tf.losses.categorical_crossentropy,metrics=['accuracy','Recall'])
                cb3 = hp.KerasCallback(logdir, {'유닛수':u, '드랍아웃':d, '옵티마이저':o})
                net.fit(X,y, epochs=4, callbacks=cb3)
                _mymetric=sum(net.evaluate(XX,yy)[1:]) # net.evaluate(XX,yy)는 [loss,accuracy,recall]를 리턴함
                tf.summary.scalar('애큐러시+리콜', _mymetric, step=1) ## 내맘대로 메트릭을 만들어서 그거 찍어보고 싶다!

2022-05-27 17:27:46.178363: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


Epoch 1/4
1875/1875 [==============================] - 3s 1ms/step - loss: 0.4893 - accuracy: 0.8271 - recall: 0.7796
Epoch 2/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.3647 - accuracy: 0.8677 - recall: 0.8427
Epoch 3/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.3263 - accuracy: 0.8808 - recall: 0.8589
Epoch 4/4
313/313 [==============================] - 0s 825us/step - loss: 0.3548 - accuracy: 0.8701 - recall: 0.8456
Epoch 1/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.7360 - accuracy: 0.7627 - recall: 0.6001
Epoch 2/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.5090 - accuracy: 0.8274 - recall: 0.7618
Epoch 3/4
1875/1875 [==============================] - 2s 1ms/step - loss: 0.4646 - accuracy: 0.8409 - recall: 0.7906
Epoch 4/4
313/313 [==============================] - 0s 868us/step - loss: 0.4682 - accuracy: 0.8342 - recall: 0.7939
Epoch 1/4
1875/1875 [==============================] - 2

In [11]:
%load_ext tensorboard
%tensorboard --logdir logs --host 0.0.0.0

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
